# Day 3 — Cloud Deployment (Render / Railway / Fly / HF Spaces)

---

Your image works. It's in GHCR. Now let's actually put it on the internet.

We'll compare four fresher-friendly platforms and deploy to **Render** step-by-step. The concepts transfer everywhere.


## 1. The 2026 fresher deployment cheat sheet

| Platform | Best for | Free tier | Deploy from |
|---|---|---|---|
| **Render** | FastAPI + Docker apps | Yes (auto-sleeps) | GitHub repo or Docker image |
| **Railway** | Same idea, nicer UX | $5 monthly credit | Git repo |
| **Fly.io** | Global edge deploys | Small always-on VM | `flyctl deploy` |
| **Hugging Face Spaces** | ML demos with Gradio/Streamlit | Yes, unlimited | Git push to HF |
| **AWS / GCP / Azure** | Real production, big spend | Complex free tiers | Many ways |

**Fresher recommendation: start with Render.** Simple, cheap, real-Linux, official Docker support, generous free tier.


## 2. Deploying to Render — the 5-step recipe

1. Sign up at https://render.com (GitHub login).
2. **New +** → **Web Service**.
3. Connect your GitHub repo.
4. Runtime: **Docker** (Render reads your Dockerfile).
5. Add environment variables (your API keys) in the Environment tab.

Render auto-deploys every push to `main`. That's it.

**Costs on free tier:** free web service that sleeps after 15 min idle (wakes on first request). Fine for demos, not for production.


## 3. `render.yaml` — deploy as code

Instead of clicking, commit a `render.yaml` at repo root:

```yaml
services:
  - type: web
    name: my-rag-app
    env: docker
    plan: free
    envVars:
      - key: TOGETHER_API_KEY
        sync: false          # you set this in the dashboard
    healthCheckPath: /healthz
```

Render reads this on every push and reconciles. Same idea as Kubernetes manifests, way simpler.


## 4. `fly.toml` — the Fly.io equivalent

If you'd rather use Fly.io (fantastic for global apps, small always-on VMs):

```toml
app = "my-rag-app"
primary_region = "iad"

[build]
  dockerfile = "Dockerfile"

[env]
  PORT = "8000"

[http_service]
  internal_port = 8000
  force_https = true
  auto_stop_machines = true
  min_machines_running = 0

[[vm]]
  memory = "512mb"
```

Deploy with `flyctl deploy`. Set secrets with `flyctl secrets set TOGETHER_API_KEY=...`.


## 5. Hugging Face Spaces — for demos with a UI

Spaces are Git repos hosted by HF that auto-deploy Gradio or Streamlit apps.

Minimal `app.py`:
```python
import gradio as gr

def chat(msg):
    return f"You said: {msg}"

gr.ChatInterface(chat).launch()
```

Push it to a Space repo → live URL in 60 seconds. Free CPU-only, or ~$10/month for a small GPU.

**Perfect for portfolio.** Slap your Section 6 or 7 project behind Gradio → shareable link.


## 6. Health checks — the underrated basic

Every deployment platform pings a URL to check if your app is alive. Add this to your FastAPI:

```python
@app.get("/healthz")
def health():
    return {"ok": True}
```

Configure the platform to hit `/healthz`. If your app crashes on startup (missing env var, DB unreachable), the platform will spot it instantly instead of routing traffic to a broken pod.


## 7. HTTPS, custom domains, and CORS

- **HTTPS**: automatic on all four platforms above. You don't do anything.
- **Custom domains**: point a CNAME at the platform-provided hostname. All four have docs.
- **CORS**: if a browser app calls your API from a different origin, add:
  ```python
  from fastapi.middleware.cors import CORSMiddleware
  app.add_middleware(CORSMiddleware, allow_origins=["https://mysite.com"], allow_methods=["*"])
  ```


## 8. AWS / GCP / Azure — when to graduate

The three "big" clouds are worth learning **once you're paid to.** For personal projects, they're overkill. Fresher checklist for when you're ready:

- Deploy the same Docker image to **ECS Fargate** (AWS) or **Cloud Run** (GCP) — both are "container-in, URL-out" services and behave like Render.
- Use **AWS App Runner** or **Google Cloud Run** as gentle onramps. Skip Kubernetes until you have a reason.
- Terraform > click-ops as soon as you have >1 environment.


## Recap

- Fresher-friendly cloud deploys: **Render** (default), **Railway**, **Fly.io**, **HF Spaces** (for Gradio).
- **`render.yaml` / `fly.toml`** — deploy-as-code. Commit next to your Dockerfile.
- Always add a **`/healthz` endpoint** so the platform can detect crashes.
- HTTPS is automatic on all fresher-tier platforms.
- Graduate to AWS/GCP/Azure once you have a *reason* — not for the resume.
- **Next class:** observability — knowing what your deployed app is actually doing.
